In [1]:
from pathlib import Path

files = [
    "collection.tar.gz",
    "queries.dev.small.tsv",
    "qrels.dev.small.tsv",
]

for file in files:
    p = Path(file)

    print("=" * 60)
    print("Arquivo:", p.name)
    print("Existe:", p.exists())

    if p.exists():
        print(
            "Tamanho:",
            round(p.stat().st_size / (1024**2), 2),
            "MB"
        )

Arquivo: collection.tar.gz
Existe: True
Tamanho: 987.06 MB
Arquivo: queries.dev.small.tsv
Existe: True
Tamanho: 0.28 MB
Arquivo: qrels.dev.small.tsv
Existe: True
Tamanho: 0.14 MB


In [2]:
import pandas as pd

queries = pd.read_csv(
    "queries.dev.small.tsv",
    sep="\t",
    header=None,
    names=["qid", "query"]
)

print("Queries:", len(queries))
print("Unique qids:", queries["qid"].nunique())

queries.head()

Queries: 6980
Unique qids: 6980


,qid,query
0,1048585,what is paula deen's brother
1,2,Androgen receptor define
2,524332,treating tension headaches without medication
3,1048642,what is paranoid sc
4,524447,treatment of varicose veins in legs


In [3]:
qrels = pd.read_csv(
    "qrels.dev.small.tsv",
    sep=r"\s+",
    header=None,
    names=["qid", "unused", "pid", "label"]
)

print("Rows:", len(qrels))
print("Unique qids:", qrels["qid"].nunique())

qrels.head()

Rows: 7437
Unique qids: 6980


,qid,unused,pid,label
0,300674,0,7067032,1
1,125705,0,7067056,1
2,94798,0,7067181,1
3,9083,0,7067274,1
4,174249,0,7067348,1


In [4]:
counts = (
    qrels.groupby("qid")
         .size()
         .reset_index(name="num_rel")
)

print(counts["num_rel"].describe())

print(
    counts["num_rel"]
          .value_counts()
          .sort_index()
)

count    6980.000000
mean        1.065473
std         0.287555
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: num_rel, dtype: float64
num_rel
1    6590
2     331
3      51
4       8
Name: count, dtype: int64


In [5]:
import tarfile

with tarfile.open("collection.tar.gz") as tar:
    print(tar.getnames())

['collection.tsv']


In [6]:
with tarfile.open("collection.tar.gz") as tar:
    tar.extractall("msmarco_collection")

In [1]:
import pandas as pd

queries = pd.read_csv(
    "queries.dev.small.tsv",
    sep="\t",
    header=None,
    names=["qid", "query"]
)

print("Queries:", len(queries))
queries.head()

Queries: 6980


,qid,query
0,1048585,what is paula deen's brother
1,2,Androgen receptor define
2,524332,treating tension headaches without medication
3,1048642,what is paranoid sc
4,524447,treatment of varicose veins in legs


In [2]:
qrels = pd.read_csv(
    "qrels.dev.small.tsv",
    sep=r"\s+",
    header=None,
    names=["qid", "unused", "pid", "label"]
)

print("Qrels:", len(qrels))
qrels.head()

Qrels: 7437


,qid,unused,pid,label
0,300674,0,7067032,1
1,125705,0,7067056,1
2,94798,0,7067181,1
3,9083,0,7067274,1
4,174249,0,7067348,1


In [3]:
collection = pd.read_csv(
    "collection.tsv",
    sep="\t",
    header=None,
    names=["pid", "passage"],
    nrows=1000
)

collection.head()

,pid,passage
0,0,The presence of communication amid scientific ...
1,1,The Manhattan Project and its atomic bomb help...
2,2,Essay on The Manhattan Project - The Manhattan...
3,3,The Manhattan Project was the name for a proje...
4,4,versions of each volume as well as complementa...


In [4]:
counts = (
    qrels.groupby("qid")
         .size()
         .reset_index(name="num_rel")
)

In [5]:
single_qids = set(
    counts[
        counts["num_rel"] == 1
    ]["qid"]
)

multi_qids = set(
    counts[
        counts["num_rel"] > 1
    ]["qid"]
)

print("Single:", len(single_qids))
print("Multi:", len(multi_qids))

Single: 6590
Multi: 390


In [6]:
gold_single = []

for _, row in qrels.iterrows():

    if row["qid"] not in single_qids:
        continue

    query_text = queries.loc[
        queries["qid"] == row["qid"],
        "query"
    ].iloc[0]

    gold_single.append({
        "query_id": int(row["qid"]),
        "query": query_text,
        "positive_ids": [
            int(row["pid"])
        ]
    })

print(len(gold_single))

6590


In [7]:
gold_multi = []

for qid in multi_qids:

    rel_docs = (
        qrels[
            qrels["qid"] == qid
        ]["pid"]
        .tolist()
    )

    query_text = queries.loc[
        queries["qid"] == qid,
        "query"
    ].iloc[0]

    gold_multi.append({
        "query_id": int(qid),
        "query": query_text,
        "positive_ids": [
            int(x)
            for x in rel_docs
        ]
    })

print(len(gold_multi))

390


In [8]:
import json

with open(
    "official_gold_single.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gold_single,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    "official_gold_multi.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gold_multi,
        f,
        indent=2,
        ensure_ascii=False
    )